In [ ]:
# ==============================================================================
# PROGETTO: Forensic Network Analysis: The Epstein Corpus
# NOTEBOOK 2: Semantic Classification via Zero-Shot NLP
# OBIETTIVI: 
# 1. Risoluzione dell'ambiguità semantica degli archi relazionali.
# 2. Implementazione del modello NLI BART-large per classificazione Zero-Shot.
# 3. Stratificazione del network (Logistico, Sociale, Legale) per RQ1 e RQ3.
# ==============================================================================

import pandas as pd
import torch
import logging
from transformers import pipeline
from tqdm.auto import tqdm
import os

# --- 0. CONFIGURAZIONE LOGGING ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# --- 1. CONFIGURAZIONE HARDWARE (Apple Silicon Optimization) ---
# Ottimizzazione specifica per architettura ARM (M2) tramite Metal Performance Shaders.
if torch.backends.mps.is_available():
    device = torch.device("mps")
    hf_device = "mps"
    logging.info("Hardware Setup: Apple Silicon (M2) rilevato. Accelerazione MPS abilitata.")
else:
    device = torch.device("cpu")
    hf_device = -1
    logging.warning("Hardware Setup: MPS non disponibile. Fallback su CPU.")

# --- 2. INIZIALIZZAZIONE PIPELINE NLP ---
logging.info("Inizializzazione del modello facebook/bart-large-mnli in corso...")
classifier = pipeline(
    "zero-shot-classification", 
    model="facebook/bart-large-mnli", 
    device=hf_device
)
logging.info("Modello caricato in memoria con successo.")

# --- 3. PREPARAZIONE DATI E DEFINIZIONE LABEL ---
candidate_labels = [
    "logistical coordination or travel planning", 
    "social networking, personal or political communication", 
    "legal procedural, investigative or oversight action"
]

input_path = '../data/edges_base.csv'
output_path = '../data/edges_classified.csv'

if not os.path.exists(input_path):
    logging.error(f"File di input non trovato: {input_path}")
    raise FileNotFoundError(f"File non trovato: {input_path}")

df_master = pd.read_csv(input_path)

# --- 4. MOTORE DI CLASSIFICAZIONE ZERO-SHOT (FULL RUN) ---
results = []
batch_size = 32  # Ottimizzato per i 16GB del tuo Mac M2
checkpoint_step = 5000

logging.info(f"Avvio classificazione totale su {len(df_master)} righe...")

for i in tqdm(range(0, len(df_master), batch_size), desc="Inferenza NLP"):
    batch = df_master.iloc[i : i + batch_size]
    
    # Costruzione del SUPER-PROMPT riga per riga con i Tags
    texts_to_classify = []
    for _, row in batch.iterrows():
        tags_raw = str(row.get('content_tags', ''))
        
        # Pulizia del JSON per rendere i tag leggibili dall'AI
        if tags_raw and tags_raw != 'nan':
            tags_clean = tags_raw.replace('[', '').replace(']', '').replace('"', '').replace(',', ', ')
        else:
            tags_clean = 'none'
        
        testo = f"The interaction is '{row['actor']} {row['action']} {row['target']}'. Document context: {row['one_sentence_summary']}. Document tags: {tags_clean}."
        texts_to_classify.append(testo)
    
    try:
        # Inferenza sul modello
        batch_predictions = classifier(texts_to_classify, candidate_labels, truncation=True)
        
        for pred in batch_predictions:
            results.append({
                'semantic_category': pred['labels'][0],
                'confidence': pred['scores'][0]
            })
    except Exception as e:
        logging.error(f"Fallimento al batch {i}. Dettagli: {str(e)}")
        # Placeholder in caso di errore per non sfalsare l'allineamento
        for _ in range(len(batch)):
            results.append({'semantic_category': 'ERROR', 'confidence': 0.0})
            
    # CHECKPOINTING: Salvataggio automatico ogni 5000 righe
    if (i + batch_size) % checkpoint_step < batch_size and i > 0:
        logging.info(f"Salvataggio intermedio: {len(results)} righe elaborate.")
        temp_df = pd.DataFrame(results)
        df_partial = pd.concat([df_master.iloc[:len(temp_df)].reset_index(drop=True), temp_df], axis=1)
        df_partial.to_csv(output_path, index=False)

# --- 5. CONSOLIDAMENTO E SALVATAGGIO FINALE ---
df_results = pd.DataFrame(results)
df_final = pd.concat([df_master.reset_index(drop=True), df_results], axis=1)

df_final.to_csv(output_path, index=False)
logging.info(f"Classificazione completata! File salvato in: {output_path}")

2026-03-13 15:35:17 - INFO - Hardware Setup: Apple Silicon (M2) rilevato. Accelerazione MPS abilitata.
2026-03-13 15:35:17 - INFO - Inizializzazione del modello facebook/bart-large-mnli in corso...
2026-03-13 15:35:17 - INFO - HTTP Request: HEAD https://huggingface.co/facebook/bart-large-mnli/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-13 15:35:17 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/bart-large-mnli/d7645e127eaf1aefc7862fd59a17a5aa8558b8ce/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

2026-03-13 15:35:17 - INFO - HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-mnli/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-13 15:35:18 - INFO - HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-mnli/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-13 15:35:20 - INFO - Modello caricato in memoria con successo.
2026-03-13 15:35:20 - INFO - Avvio classificazione totale su 107021 righe...


Inferenza NLP:   0%|          | 0/3345 [00:00<?, ?it/s]